# GroupBy
> `MXGroupBy` — group rows by one or more keys, then aggregate with MAX Engine.
>
> **CPU path** (`device='cpu'`): one compiled `max_masked_sum` graph per aggregation column,
> cached and reused. Correct for any schema.
>
> **GPU path** (`device='gpu'`): detects the TPC-H Q1 fused aggregation pattern and dispatches
> to `gpu_fused_group_agg()` — a single Mojo kernel launch that computes 6 aggregations for
> one group in one pass over the data.  Falls back to `gpu_masked_sum_kernel()` per column
> for any non-Q1 schema.
>
> **Group-id column** (`_build_group_ids`): produces a compact `int32` array mapping each row
> to its 0-based group index.  Prerequisite for the generic `group_sum` kernel in v0.0.4.

In [1]:
#| default_exp groupby

In [2]:
#| export
from __future__ import annotations

import numpy as np
import pyarrow as pa
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional, Literal, Set

from mxframe.core_bridge import MXFrame, DeviceType
from mxframe.max_engine import (
    max_masked_sum, max_min, max_max, max_count,
    gpu_masked_sum_kernel, gpu_fused_group_agg,
    group_sum_kernel,
    kernels_available, _FUSED_AGG_COLS,
)


## Aggregation expression

In [3]:
#| export
@dataclass
class AggExpr:
    """Thin descriptor of one aggregation: `(column_name, op)`."""
    column: str
    op:     Literal["sum", "mean", "count", "min", "max"]


def agg_sum(column: str)   -> AggExpr: return AggExpr(column, "sum")
def agg_mean(column: str)  -> AggExpr: return AggExpr(column, "mean")
def agg_count(column: str) -> AggExpr: return AggExpr(column, "count")
def agg_min(column: str)   -> AggExpr: return AggExpr(column, "min")
def agg_max(column: str)   -> AggExpr: return AggExpr(column, "max")

## MXGroupBy

In [4]:
#| export
class MXGroupBy:
    """
    Groups an `MXFrame` by one or more key columns and aggregates with MAX Engine.

    **v0.0.4 — single-pass scatter-sum** (one kernel call per agg column)

    Execution paths, chosen automatically at runtime:

    - **Scatter path** (`kernels_available() and n_groups ≤ 64`):
      `group_sum_kernel(col, group_ids, n_groups, device)` processes ALL groups
      in a single O(N) pass.  `sum` and `count` cost 1 kernel call each;
      `mean` = 2 calls;  `min`/`max` fall back to numpy (no Mojo kernel yet).
      **Total: O(n_agg_cols) kernel calls** instead of O(n_groups × n_agg_cols).

    - **Per-group mask fallback** (kernels unavailable or n_groups > 64):
      `max_masked_sum()` via MAX Graph API, compiled + cached per shape.
      Same correctness guarantee, slower due to repeated O(N) passes.

    Examples:
        >>> gb = MXGroupBy(df, by=["returnflag", "linestatus"], device="cpu")
        >>> result = gb.agg(
        ...     sum_qty=agg_sum("l_quantity"),
        ...     count_order=agg_count("l_quantity"),
        ... )
    """

    def __init__(
        self,
        frame:  MXFrame,
        by:     List[str],
        device: DeviceType = "cpu",
    ):
        self._frame  = frame
        self._by     = by if isinstance(by, list) else [by]
        self._device = device
        self._masks:     Optional[Dict[Tuple, np.ndarray]] = None
        self._group_ids: Optional[np.ndarray]              = None
        self._group_keys: Optional[List[Tuple]]            = None

    # ------------------------------------------------------------------
    # Internal: build boolean masks  (O(N) pass, built once)
    # ------------------------------------------------------------------

    def _build_masks(self) -> None:
        if self._masks is not None:
            return
        table    = self._frame.to_arrow()
        n        = table.num_rows
        key_cols = [table.column(k).to_pylist() for k in self._by]

        groups: Dict[Tuple, List[int]] = {}
        for i in range(n):
            key = tuple(c[i] for c in key_cols)
            if key not in groups:
                groups[key] = []
            groups[key].append(i)

        self._group_keys = sorted(groups.keys())
        self._masks      = {}
        for key, idxs in groups.items():
            m        = np.zeros(n, dtype=np.float32)
            m[idxs]  = 1.0
            self._masks[key] = m

    # ------------------------------------------------------------------
    # Internal: build compact group-id array  (v0.0.4 scatter path)
    # ------------------------------------------------------------------

    def _build_group_ids(self) -> np.ndarray:
        """
        Return a `[N]` int32 array where `group_ids[i]` is the 0-based index
        of the group that row `i` belongs to, ordered by sorted group key.
        """
        if self._group_ids is not None:
            return self._group_ids
        self._build_masks()
        n   = self._frame.num_rows
        ids = np.full(n, -1, dtype=np.int32)
        for gi, key in enumerate(self._group_keys):
            ids[self._masks[key].astype(bool)] = gi
        self._group_ids = ids
        return ids

    # ------------------------------------------------------------------
    # Fallback: per-group mask aggregation  (O(groups × N))
    # ------------------------------------------------------------------

    def _agg_all_groups_masked(
        self,
        value_arrs: Dict[str, np.ndarray],
        kwargs:     Dict[str, "AggExpr"],
    ) -> Dict[str, list]:
        """
        Compute aggregations via per-group masks — used when the scatter path
        is unavailable (no kernels.mojopkg or n_groups > 64).
        """
        out_vals: Dict[str, list] = {k: [] for k in kwargs}
        for key in self._group_keys:
            mask     = self._masks[key]
            mask_sum = float(mask.sum())
            for out_name, expr in kwargs.items():
                vals = value_arrs[expr.column]
                if expr.op == "sum":
                    out_vals[out_name].append(
                        max_masked_sum(mask, vals, device=self._device)
                    )
                elif expr.op == "mean":
                    s = max_masked_sum(mask, vals, device=self._device)
                    out_vals[out_name].append(s / mask_sum if mask_sum > 0 else float("nan"))
                elif expr.op == "count":
                    out_vals[out_name].append(int(mask_sum))
                elif expr.op == "min":
                    gv = vals[mask.astype(bool)]
                    out_vals[out_name].append(float(gv.min()) if len(gv) else float("nan"))
                elif expr.op == "max":
                    gv = vals[mask.astype(bool)]
                    out_vals[out_name].append(float(gv.max()) if len(gv) else float("nan"))
                else:
                    raise ValueError(f"Unknown op: {expr.op!r}")
        return out_vals

    # ------------------------------------------------------------------
    # Fast path: scatter-sum all groups in one kernel call per column
    # ------------------------------------------------------------------

    def _agg_all_groups_scatter(
        self,
        value_arrs: Dict[str, np.ndarray],
        kwargs:     Dict[str, "AggExpr"],
        gids:       np.ndarray,
        n_grps:     int,
    ) -> Dict[str, list]:
        """
        Compute aggregations using `group_sum_kernel` — one O(N) pass per column.

        Counts are computed lazily and cached across `count` and `mean` ops.
        `min`/`max` fall back to numpy boolean-slice (no Mojo kernel yet).
        """
        dev = self._device
        out_vals: Dict[str, list] = {}

        # Lazy count array (reused by both 'count' and 'mean' ops)
        _counts_cache: Optional[np.ndarray] = None

        def get_counts() -> np.ndarray:
            nonlocal _counts_cache
            if _counts_cache is None:
                ones = np.ones(len(gids), dtype=np.float32)
                _counts_cache = group_sum_kernel(ones, gids, n_grps, device=dev)
            return _counts_cache

        for out_name, expr in kwargs.items():
            col = value_arrs[expr.column]
            if expr.op == "sum":
                out_vals[out_name] = group_sum_kernel(col, gids, n_grps, device=dev).tolist()

            elif expr.op == "mean":
                sums   = group_sum_kernel(col, gids, n_grps, device=dev)
                counts = get_counts()
                safe_c = np.where(counts > 0, counts, 1.0)
                out_vals[out_name] = (sums / safe_c).tolist()

            elif expr.op == "count":
                out_vals[out_name] = [int(x) for x in get_counts().tolist()]

            elif expr.op in ("min", "max"):
                # No Mojo scatter kernel for min/max yet — numpy per-group fallback
                vals_list = []
                for gi in range(n_grps):
                    group_col = col[gids == gi]
                    if len(group_col) == 0:
                        vals_list.append(float("nan"))
                    elif expr.op == "min":
                        vals_list.append(float(group_col.min()))
                    else:
                        vals_list.append(float(group_col.max()))
                out_vals[out_name] = vals_list

            else:
                raise ValueError(f"Unknown op: {expr.op!r}")

        return out_vals

    # ------------------------------------------------------------------
    # Public: agg()
    # ------------------------------------------------------------------

    def agg(self, **kwargs: "AggExpr") -> MXFrame:
        """
        Compute named aggregations for each group.

        Returns an `MXFrame` with key columns first, then one column per agg.

        Args:
            **kwargs: `output_name=AggExpr(...)` pairs, e.g.
                      `total=agg_sum("price"), n=agg_count("qty")`

        Returns:
            MXFrame with one row per group and one column per aggregation.
        """
        self._build_group_ids()    # also calls _build_masks()
        gids   = self._group_ids
        n_grps = len(self._group_keys)
        table  = self._frame.to_arrow()

        # Pre-load all needed columns as contiguous float32 numpy arrays
        value_arrs: Dict[str, np.ndarray] = {}
        for expr in kwargs.values():
            if expr.column not in value_arrs:
                arr = (
                    table.column(expr.column)
                    .to_numpy(zero_copy_only=False)
                    .astype(np.float32)
                )
                value_arrs[expr.column] = np.ascontiguousarray(arr)

        # Choose execution path
        use_scatter = kernels_available() and n_grps <= 64

        if use_scatter:
            out_vals = self._agg_all_groups_scatter(value_arrs, kwargs, gids, n_grps)
        else:
            out_vals = self._agg_all_groups_masked(value_arrs, kwargs)

        # Build key columns (sorted order matches _group_keys)
        out_keys: Dict[str, list] = {k: [] for k in self._by}
        for key in self._group_keys:
            for k, v in zip(self._by, key):
                out_keys[k].append(v)

        return MXFrame(pa.table({**out_keys, **out_vals}))

    def __repr__(self) -> str:
        dev    = self._device
        n_grps = len(self._group_keys) if self._group_keys else "?"
        path   = ("scatter" if kernels_available() and (
            isinstance(n_grps, int) and n_grps <= 64) else "masked")
        return (
            f"MXGroupBy(by={self._by}, device={dev}, "
            f"rows={self._frame.num_rows}, groups={n_grps}, path={path})"
        )


## 🔀 v0.0.4 Execution Paths

```
MXGroupBy.agg(device='cpu' or 'gpu')
         │
         │  Build group_ids [N] int32  (single O(N) pass)
         │
         ├─ FAST SCATTER PATH  (kernels_available() AND n_groups ≤ 64)
         │        │
         │        │  For each aggregation column:
         │        │
         │        ├─ sum  ─────  group_sum_kernel(col, gids, G, device)
         │        │                └─ 1 kernel call → all G group sums  O(N)
         │        │
         │        ├─ count ────  group_sum_kernel(ones, gids, G, device)
         │        │                └─ 1 kernel call, result cached across ops
         │        │
         │        ├─ mean ─────  sum_kernel + count_kernel / O(G) division
         │        │                └─ 2 kernel calls (count cached)
         │        │
         │        └─ min/max ──  numpy per-group boolean slice   ← no Mojo kernel yet
         │
         │  Total kernel calls: O(n_agg_cols)  instead of O(G × n_agg_cols)
         │  Speedup over v0.0.3: ~n_groups×  (6× for Q1, where G=6)
         │
         └─ MASKED FALLBACK  (no kernels.mojopkg OR n_groups > 64)
                  │
                  │  For each group, for each agg column:
                  │    max_masked_sum(mask, col, device)  ← Graph API, cached per shape
                  │    min/max: numpy boolean slice
                  │
                  └─ Total kernel calls: O(G × n_agg_cols)  — slower but always correct

**📐 Kernel call comparison (Q1 benchmark: 6 groups, 8 agg columns)**

| Version | Strategy | Kernel calls | Expected time |
|---|---|---|---|
| v0.0.3 | per-group masked sum | 48 | ~2 270ms |
| **v0.0.4** | **scatter-sum** | **8** | **~300–500ms** |
| Polars | native Rust | — | ~34ms |

The gap to Polars is now algorithmic overhead (nbdev Python layer, PyArrow I/O) rather
than O(N) kernel launch counts.


In [ ]:
import numpy as np
from mxframe.core_bridge import MXFrame
from max import driver

GPU_AVAILABLE = driver.accelerator_count() > 0
print(f"GPU available: {GPU_AVAILABLE}")
print(f"Kernels available: {kernels_available()}")

df = MXFrame({
    "group":  ["A", "B", "A", "B", "A"],
    "qty":    [1.0, 2.0, 3.0, 4.0, 5.0],
    "price":  [10.0, 20.0, 30.0, 40.0, 50.0],
})

devices_to_test = ["cpu"] + (["gpu"] if GPU_AVAILABLE else [])

for dev in devices_to_test:
    gb  = MXGroupBy(df, by=["group"], device=dev)
    res = gb.agg(
        total_qty = agg_sum("qty"),
        avg_qty   = agg_mean("qty"),
        n         = agg_count("qty"),
        min_qty   = agg_min("qty"),
        max_qty   = agg_max("qty"),
    )
    tbl = res.to_arrow()

    # Group A: rows 0,2,4  qty=[1,3,5]  sum=9  mean=3  count=3  min=1  max=5
    # Group B: rows 1,3    qty=[2,4]    sum=6  mean=3  count=2  min=2  max=4
    for i in range(tbl.num_rows):
        g    = tbl.column("group")[i].as_py()
        s    = tbl.column("total_qty")[i].as_py()
        avg  = tbl.column("avg_qty")[i].as_py()
        cnt  = tbl.column("n")[i].as_py()
        mn   = tbl.column("min_qty")[i].as_py()
        mx   = tbl.column("max_qty")[i].as_py()
        expected = {"A": (9.0, 3.0, 3, 1.0, 5.0), "B": (6.0, 3.0, 2, 2.0, 4.0)}[g]
        assert abs(s   - expected[0]) < 1e-3, f"[{dev}] {g} sum: {s}"
        assert abs(avg - expected[1]) < 1e-3, f"[{dev}] {g} mean: {avg}"
        assert cnt == expected[2],             f"[{dev}] {g} count: {cnt}"
        assert abs(mn  - expected[3]) < 1e-3, f"[{dev}] {g} min: {mn}"
        assert abs(mx  - expected[4]) < 1e-3, f"[{dev}] {g} max: {mx}"
    print(f"✅ MXGroupBy({dev}): sum/mean/count/min/max all correct")

# ── group_ids ─────────────────────────────────────────────────────────────────
gb2  = MXGroupBy(df, by=["group"], device="cpu")
ids  = gb2._build_group_ids()
print(f"\ngroup_ids (sorted groups A=0, B=1): {ids.tolist()}")
# A appears at 0,2,4 → id=0  ;  B at 1,3 → id=1
assert ids[0] == ids[2] == ids[4], "A rows should share same group id"
assert ids[1] == ids[3],           "B rows should share same group id"
assert ids[0] != ids[1],           "A and B must have different ids"
print("✅ group_ids correct")

print("\n✅ All GroupBy tests passed")

GPU available: True
Kernels available: True
✅ MXGroupBy(cpu): sum/mean/count/min/max all correct
✅ MXGroupBy(gpu): sum/mean/count/min/max all correct

group_ids (sorted groups A=0, B=1): [0, 1, 0, 1, 0]
✅ group_ids correct

✅ All GroupBy tests passed
